# NOVA × 魔搭 A10 24GB

第一条真实成片纵切：已授权人物图片 + WAV → EchoMimicV3 Flash MP4。请按顺序运行，完成后停止 Notebook 实例。

In [ ]:
import shutil, subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
total, used, free = shutil.disk_usage('/mnt/workspace')
print('持久盘可用 GiB:', round(free / 1024**3, 1))
assert free >= 45 * 1024**3, '至少需要 45GiB 可用空间'

In [ ]:
%pip install -U modelscope-hub

In [ ]:
from pathlib import Path
BUNDLE_DIR = Path('/mnt/workspace/NOVA-ModelScope-A10')
assert (BUNDLE_DIR / 'prepare_modelscope.py').is_file(), '请把整个 deploy/modelscope 目录上传到指定位置'
print('部署包:', BUNDLE_DIR)

In [ ]:
!python {BUNDLE_DIR / 'prepare_modelscope.py'} --root /mnt/workspace/nova

In [ ]:
!bash {BUNDLE_DIR / 'bootstrap_runtime.sh'} /mnt/workspace/nova

In [ ]:
IMAGE = Path('/mnt/workspace/nova-inputs/avatar.png')
AUDIO = Path('/mnt/workspace/nova-inputs/voice.wav')
assert IMAGE.is_file(), f'缺少 {IMAGE}'
assert AUDIO.is_file(), f'缺少 {AUDIO}'
print('输入已就绪')

In [ ]:
!python {BUNDLE_DIR / 'run_flash.py'} --image {IMAGE} --audio {AUDIO} --root /mnt/workspace/nova --frames 81

In [ ]:
from IPython.display import Video, display
outputs = sorted(Path('/mnt/workspace/nova/outputs').glob('*_output.mp4'), key=lambda p: p.stat().st_mtime)
assert outputs, '没有找到输出 MP4，请检查上一个单元格日志'
print(outputs[-1])
display(Video(str(outputs[-1]), embed=False))